<a href="https://colab.research.google.com/github/kumarmohit0911/robust-gas-classification-mq-sensors/blob/main/hyperparam_tuning_using_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.4 MB/s eta 0:00:00


In [5]:
# calling all the dependencies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [6]:
#for reproducibility
torch.manual_seed(42)

In [7]:
# loading the data set
df = pd.read_csv("/content/Gas_Sensors_Measurements (2).csv")

In [8]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [9]:
df = df.drop(["Serial Number","Corresponding Image Name"],axis = 1)


In [10]:
for i in df['Gas']:
  if i == 'NoGas':
    df['Gas'] = df['Gas'].replace(i,0)
  if i == 'Perfume':
    df['Gas'] = df['Gas'].replace(i,1)
  if i == 'Smoke':
    df['Gas'] = df['Gas'].replace(i,2)
  if i == 'Mixture':
    df['Gas'] = df['Gas'].replace(i,3)

/tmp/ipython-input-512672298.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Gas'] = df['Gas'].replace(i,3)


In [11]:
X = df[['MQ2','MQ3','MQ5', 'MQ6',	'MQ7',	'MQ8',	'MQ135']]
y = df['Gas']

In [12]:
# scaling the features

sc = StandardScaler()
X = sc.fit_transform(X)

In [13]:
# Train test Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [14]:
# creating custom dataset

class CustomDataset(Dataset):
  def __init__(self,features, labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [15]:
train_dataset = CustomDataset(X_train,y_train.values)
test_dataset = CustomDataset(X_test,y_test.values)

In [16]:
# Creating train and test dataloader
train_loader = DataLoader(train_dataset,batch_size =  32, shuffle =True)
test_loader = DataLoader(test_dataset,batch_size = 32,shuffle = True)

In [17]:
class MyNN(nn.Module):
  def __init__(self, input_dim, output_dim,
               num_hidden_layer, neurons_per_layer):
    super().__init__()
    layers = []
    for i in range(num_hidden_layer):
      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU()),
      layers.append(nn.Dropout(p=0.3))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(neurons_per_layer,output_dim))
    self.model = nn.Sequential(*layers)
  def forward(self,x):
    return self.model(x)

In [18]:
# ready the objective function
def objective(trial):
  # hyperparameter value for search space
  num_hidden_layer = trial.suggest_int("num_hidden_layers",1,5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)
  #model initialisation
  input_dim = 7
  output_dim = 4
  model = MyNN(input_dim,output_dim,num_hidden_layer,neurons_per_layer)
  model.to(device)
  #parameter initialisation
  learning_rate = 0.01
  epochs = 200
  #optimmizer selectiom
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(),lr = learning_rate,weight_decay = 1e-4)
  #training loop

  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features,batch_labels in train_loader:

      #modifying training loop data to gpu
      batch_features,batch_labels = batch_features.to(device), batch_labels.to(device)


      #forward pass
      outputs = model(batch_features)

      #calculate loss
      loss = criterion(outputs,batch_labels)

      #backward pass
      optimizer.zero_grad()
      loss.backward()

      #update weights
      optimizer.step()

  #evaluation
  model.eval()
  #evaluation code over test data
  #Test Code
  total = 0
  correct = 0
  with torch.no_grad():
    for batch_features,batch_labels in test_loader:
      batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
      outputs = model(batch_features)
      _,predicted = torch.max(outputs.data,1)
      total += batch_labels.shape[0]
      correct += (predicted == batch_labels).sum().item()

  accuracy = correct/total
  return accuracy

In [19]:
import optuna
study = optuna.create_study(direction = 'maximize')


[I 2026-01-02 16:49:39,059] A new study created in memory with name: no-name-be939ceb-816c-44e6-9da4-bc080f9b69e7


In [20]:
study.optimize(objective, n_trials= 10)

[I 2026-01-02 16:51:03,529] Trial 0 finished with value: 0.93515625 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 88}. Best is trial 0 with value: 0.93515625.
[I 2026-01-02 16:52:19,083] Trial 1 finished with value: 0.93828125 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 32}. Best is trial 1 with value: 0.93828125.
[I 2026-01-02 16:53:46,267] Trial 2 finished with value: 0.934375 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 24}. Best is trial 1 with value: 0.93828125.
[I 2026-01-02 16:54:35,850] Trial 3 finished with value: 0.93203125 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 72}. Best is trial 1 with value: 0.93828125.
[I 2026-01-02 16:56:03,683] Trial 4 finished with value: 0.9421875 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 64}. Best is trial 4 with value: 0.9421875.
[I 2026-01-02 16:57:18,332] Trial 5 finished with value: 0.9390625 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 

In [21]:
study.best_value

0.94375

In [22]:
study.best_params

{'num_hidden_layers': 1, 'neurons_per_layer': 24}